# Coupled solver example


In [4]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *   # Opencascade for geometry modeling
import pandas as pd
import matplotlib.pyplot as plt

outer = Rectangle(2, 2).Face()
outer.edges.name="outer"
outer.edges.Max(X).name = "r"
outer.edges.Min(X).name = "l"
outer.edges.Min(Y).name = "b"
outer.edges.Max(Y).name = "t"

inner = MoveTo(0.5, 0.5).Rectangle(1, 1).Face()
inner.edges.name="interface"
outer = outer - inner

inner.faces.name="inner"
inner.faces.col = (1, 0, 0)
outer.faces.name="outer"

geo = Glue([inner, outer])

mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.1))
Draw(mesh);

'''
Definition of the time parameters
'''
T = 0.1
dt = Parameter(0.001)
t = Parameter(0)

'''
Definition of the simulation
'''
from cosmos.solvers.simulation import MovingSimulation
simulation = MovingSimulation(mesh=mesh, dt=dt, t=t, T=T)

# Adding Willmore flow to the internal square
from cosmos.solvers.willmore import StabWillmoreSolver
displ_solver = StabWillmoreSolver(domain = 'interface')
displ_solver.SaveSolution(folderpath = './coupled_results/', filename = 'coupled_displ', n_steps = 10)

# Adding the Willmore flow motion to the overall simulation
simulation.AddMotionSolver(displ_solver)
def displ():
    return displ_solver.get_solution()[0]
simulation.AddMotion(function=displ, domain = 'interface')

# Adding ADRsolvers in the different domains
from cosmos.solvers.adr import ADRSolver
from cosmos.solvers.tools import gradient

adr_solver = ADRSolver(linear = True)
simulation.AddSolver(adr_solver)


# Solver 1
u_ex1 = cos(pi*x)*sin(pi*y)*cos(t)
d1 = 1 + x**2
c1 = 1 + x**2
b1 = CF((sin(x), 1))
flux1 = b1*u_ex1 - d1*gradient(u_ex1, Id(2))
rhs1 = (u_ex1.Diff(t) + Trace(gradient(flux1, Id(2))) + c1*u_ex1).Compile()
dir_b1 = {'interface': u_ex1}
dir_d1 = {'interface': u_ex1}
neu_b1 = {'interface': b1*u_ex1}
neu_d1 = {'interface': - d1*gradient(u_ex1, Id(2))}
adr_solver.AddSpecie(VorB = VOL,
                rhs = rhs1,
                advection = b1,
                diffusion = d1,
                reaction = c1,
                u0 = u_ex1,
                neu_d = neu_d1,
                neu_b = neu_b1,
                domain = 'inner')

# Solver 1
u_ex2 = cos(pi*x)*cos(t)
d2 = 1 + x**2
c2 = 1 + t**2
b2 = CF((sin(x), 2))
flux2 = b2*u_ex2 - d2*gradient(u_ex2, Id(2))
rhs2 = (u_ex2.Diff(t) + Trace(gradient(flux2, Id(2))) + c2*u_ex2).Compile()
dir_b2 = {'interface': u_ex2}
dir_d2 = {'interface': u_ex2}
neu_b2 = {'r|l|t|b': b2*u_ex2}
neu_d2 = {'r|l|t|b': - d2*gradient(u_ex2, Id(2))}
adr_solver.AddSpecie(VorB = VOL,
                rhs = rhs2,
                advection = b2,
                diffusion = d2,
                reaction = c2,
                u0 = u_ex2,
                neu_d = neu_d2,
                neu_b = neu_b2,
                dir_d = dir_d2,
                dir_b = dir_b2,
                domain = 'outer')

adr_solver.SaveSolution(folderpath = './coupled_results/', filename = 'coupled_adr', n_steps = 10)

simulation.run()

adr_solver.draw_solution()
displ_solver.draw_solution()

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [5]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *   # Opencascade for geometry modeling
import pandas as pd
import matplotlib.pyplot as plt

outer = Rectangle(2, 2).Face()
outer.edges.name="outer"
outer.edges.Max(X).name = "r"
outer.edges.Min(X).name = "l"
outer.edges.Min(Y).name = "b"
outer.edges.Max(Y).name = "t"

inner = Circle((1, 1), 0.5).Face()
inner.edges.name="interface"
outer = outer - inner

inner.faces.name="inner"
inner.faces.col = (1, 0, 0)
outer.faces.name="outer"

geo = Glue([inner, outer])

mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.1))
Draw(mesh);

'''
Definition of the time parameters
'''
T = 0.018
dt = Parameter(0.001)
t = Parameter(0)

'''
Definition of the simulation
'''
from cosmos.solvers.simulation import MovingSimulation
simulation = MovingSimulation(mesh=mesh, dt=dt, t=t, T=T)

# Adding Willmore flow to the internal square
from cosmos.solvers.mean_curvature import MeanCurvatureSolver
displ_solver = MeanCurvatureSolver(domain = 'interface')
displ_solver.SaveSolution(folderpath = './coupled_results/', filename = 'coupled_displ', n_steps = 10)

# Adding the Willmore flow motion to the overall simulation
simulation.AddMotionSolver(displ_solver)
def displ():
    return displ_solver.get_solution()[0]
simulation.AddMotion(function=displ, domain = 'interface')

# Adding ADRsolvers in the different domains
from cosmos.solvers.adr import ADRSolver
from cosmos.solvers.tools import gradient

adr_solver = ADRSolver(linear = True)
simulation.AddSolver(adr_solver)


# Solver 1
u_ex1 = cos(pi*x)*sin(pi*y)*cos(t)
d1 = 1 + x**2
c1 = 1 + x**2
b1 = CF((sin(x), 1))
flux1 = b1*u_ex1 - d1*gradient(u_ex1, Id(2))
rhs1 = (u_ex1.Diff(t) + Trace(gradient(flux1, Id(2))) + c1*u_ex1).Compile()
dir_b1 = {'interface': u_ex1}
dir_d1 = {'interface': u_ex1}
neu_b1 = {'interface': b1*u_ex1}
neu_d1 = {'interface': - d1*gradient(u_ex1, Id(2))}
adr_solver.AddSpecie(VorB = VOL,
                rhs = rhs1,
                advection = b1,
                diffusion = d1,
                reaction = c1,
                u0 = u_ex1,
                neu_d = neu_d1,
                neu_b = neu_b1,
                domain = 'inner')

# Solver 1
u_ex2 = cos(pi*x)*cos(t)
d2 = 1 + x**2
c2 = 1 + t**2
b2 = CF((sin(x), 2))
flux2 = b2*u_ex2 - d2*gradient(u_ex2, Id(2))
rhs2 = (u_ex2.Diff(t) + Trace(gradient(flux2, Id(2))) + c2*u_ex2).Compile()
dir_b2 = {'interface': u_ex2}
dir_d2 = {'interface': u_ex2}
neu_b2 = {'r|l|t|b': b2*u_ex2}
neu_d2 = {'r|l|t|b': - d2*gradient(u_ex2, Id(2))}
adr_solver.AddSpecie(VorB = VOL,
                rhs = rhs2,
                advection = b2,
                diffusion = d2,
                reaction = c2,
                u0 = u_ex2,
                neu_d = neu_d2,
                neu_b = neu_b2,
                dir_d = dir_d2,
                dir_b = dir_b2,
                domain = 'outer')

adr_solver.SaveSolution(folderpath = './coupled_results/', filename = 'coupled_adr', n_steps = 10)

simulation.run()

adr_solver.draw_solution()
displ_solver.draw_solution()

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…